Time to practice my text synthesis skills. We should do some sort of analysis on the effects that go in a deck. 

In [ ]:

from collections import defaultdict as dd
from os import path 

import src.card_data_collection_pipeline
import src.db_operations
import src.helpers

import nltk
import pandas as pd

# nltk.download('stopwords')
# nltk.download('averaged_perceptron_tagger')
# df = src.db_operations.from 
df = pd.read_json(path.join('.json','ROADTRIP.json'))

memoizer = dd(None)


def create_book_from_deck(deck_dict):
    """
    Combine all card effect text from a deck dictionary into a single 'book'.
    
    Args:
        deck_dict: Nested dictionary of cards with 'effect' attribute
    
    Returns:
        str: Combined effect text
    """
    effects = []
    for card_id_and_name, amount in deck_dict.items():
        id = src.helpers.extract_card_id(card_id_and_name)

        lookup = None
        if memoizer[id]:
            lookup = memoizer[id]
        if not lookup:
            lookup = src.card_data_collection_pipeline.db_lookup(card_id_and_name)
            memoizer[id] = lookup
        if not lookup:
            lookup = src.card_data_collection_pipeline.url_lookup( card_id_and_name)
            memoizer[id] = lookup

        effects.append(lookup.effect)

    book = ""
    for effect in effects:
        book += effect
    return book


def find_common_bigrams(text, top_n=10):
    """
    Find the most common bigrams in a text.
    
    Args:
        text: String of text to analyze
        top_n: Number of top bigrams to return
    
    Returns:
        list: Top N bigrams with their frequencies
    """
    # Tokenize and convert to lowercase
    tokens = nltk.word_tokenize(text.lower())
    
    # Remove stopwords
    stop_words = set('.')
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    
    # Find bigrams
    bigram_finder = nltk.QuadgramCollocationFinder.from_words(filtered_tokens)
    return bigram_finder.nbest(nltk.QuadgramAssocMeasures.raw_freq, top_n)


def analyze_decks(df, top_n=10):
    """
    Analyze bigrams across all decks in a dataframe.
    
    Args:
        df: DataFrame with 'deck' column containing nested dictionaries
        top_n: Number of top bigrams per deck
    
    Returns:
        DataFrame: Results with deck index and their top bigrams
    """
    results = []
    for idx, row in df.iterrows():
        book = create_book_from_deck(row['deck'])
        bigrams = find_common_bigrams(book, top_n)
        results.append({'deck_index': idx, 'top_bigrams': bigrams})
    
    return pd.DataFrame(results)



In [ ]:
df=df[df['name']=='Champion']

In [ ]:

results = analyze_decks(df)
print(results)

      deck_index top_bigrams
0              0          []
1              1          []
2              2          []
3              3          []
4              4          []
...          ...         ...
6349        6349          []
6350        6350          []
6351        6351          []
6352        6352          []
6353        6353          []

[6354 rows x 2 columns]
